### Feature engineering for model training

### Imports and Set up

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

epsilon = 1e-8
raw_data_path = Path.cwd().parent.parent / "data" / "raw"
processed_path = Path.cwd().parent.parent / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

file_path = os.path.join(raw_data_path, "sensor_point_dataset_raw.csv")
df = pd.read_csv(file_path)

print(f"Loaded shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

Loaded shape: (30643, 35)
Label distribution:
label
1    10500
0    10500
2     9643
Name: count, dtype: int64


### Sort

In [35]:
df = df.sort_values(["scenario_id", "timestep"], ascending=[True, True])
df = df.reset_index(drop=True)

sample_id = df["scenario_id"].iloc[0]
sample = df[df["scenario_id"] == sample_id].head(5)
print(f"Checking scenario: {sample_id}")
print(sample[["scenario_id", "timestep"]].to_string())
print("Sort verified ✓")

Checking scenario: blockage_25_loc16m
          scenario_id  timestep
0  blockage_25_loc16m         1
1  blockage_25_loc16m         2
2  blockage_25_loc16m         3
3  blockage_25_loc16m         4
4  blockage_25_loc16m         5
Sort verified ✓


### Pressure Features

In [36]:
def engineer_pressure_features(df, epsilon=1e-8):
    df = df.copy()

    df["pressure_drop_ab"] = df["node_a_pressure"] - df["node_b_pressure"]
    df["pressure_drop_bc"] = df["node_b_pressure"] - df["node_c_pressure"]
    df["pressure_drop_ac"] = df["node_a_pressure"] - df["node_c_pressure"]


    # pressure change per metre

    df["pressure_gradient_ab"] = df["pressure_drop_ab"] / 20.0
    df["pressure_gradient_bc"] = df["pressure_drop_bc"] / 20.0
    df["pressure_gradient_ac"] = df["pressure_drop_ac"] / 40.0

    df["expected_midpoint_pressure"] = (df["node_a_pressure"] + df["node_c_pressure"]) / 2

    df["midpoint_pressure_deviation"] = (df["node_b_pressure"] - df["expected_midpoint_pressure"])

    df["pressure_asymmetry"] = abs(df["pressure_drop_ab"] - df["pressure_drop_bc"])

    pressure_epsilon = 100.0  # Pa — minimum meaningful pressure

    df["pressure_ratio_ab"] = df["node_a_pressure"] / (df["node_b_pressure"] + pressure_epsilon)
    df["pressure_ratio_bc"] = df["node_b_pressure"] / (df["node_c_pressure"] + pressure_epsilon)
    df["pressure_ratio_ac"] = df["node_a_pressure"] / (df["node_c_pressure"] + pressure_epsilon)
    df["upstream_downstream_ratio"] = df["node_a_pressure"] / (df["node_c_pressure"] + pressure_epsilon)

    df["upstream_downstream_ratio"] = (df["node_a_pressure"] / (df["node_c_pressure"] + epsilon))

    if "scenario_id" not in df.columns:
        raise ValueError("scenario_id column is missing before temporal features")

    # Ensure proper ordering
    df = df.sort_values(["scenario_id", "timestep"])

    dt = 0.5

    # First derivatives
    df["dp_dt_a"] = df.groupby("scenario_id")["node_a_pressure"].diff() / dt
    df["dp_dt_b"] = df.groupby("scenario_id")["node_b_pressure"].diff() / dt
    df["dp_dt_c"] = df.groupby("scenario_id")["node_c_pressure"].diff() / dt

    # Second derivatives
    df["d2p_dt2_a"] = df.groupby("scenario_id")["dp_dt_a"].diff() / dt
    df["d2p_dt2_b"] = df.groupby("scenario_id")["dp_dt_b"].diff() / dt
    df["d2p_dt2_c"] = df.groupby("scenario_id")["dp_dt_c"].diff() / dt

    # Fill NaNs from diff()
    temporal_cols = [
        "dp_dt_a", "dp_dt_b", "dp_dt_c",
        "d2p_dt2_a", "d2p_dt2_b", "d2p_dt2_c"
    ]
    df[temporal_cols] = df[temporal_cols].fillna(0.0)

    return df


# Run it
df = engineer_pressure_features(df, epsilon)
print(f"Shape after pressure engineering: {df.shape}")

Shape after pressure engineering: (30643, 48)


- Raw pressure nodes tells us the absolute value, but they do not depict a pattern, pressure gradient does this.
- Gradient tells us how pressure changes partially along the pipe.
- Midpoint tells us if node B the middle one is behaving as expected with A and C (single most engineered feature)
- Ratios captures the relative magnitude  differences that absolute drops can miss.

### Velocity Features

In [37]:
def engineer_velocity_features(df, epsilon=1e-8):
    df = df.copy()

    df["velocity_drop_ab"] = df["velocity_a"] - df["velocity_b"]
    df["velocity_drop_bc"] = df["velocity_b"] - df["velocity_c"]
    df["velocity_drop_ac"] = df["velocity_a"] - df["velocity_c"]

    df["mean_velocity"] = df[["velocity_a", "velocity_b", "velocity_c"]].mean(axis=1)

    df["velocity_deviation_a"] = df["velocity_a"] - df["mean_velocity"]
    df["velocity_deviation_b"] = df["velocity_b"] - df["mean_velocity"]
    df["velocity_deviation_c"] = df["velocity_c"] - df["mean_velocity"]

    df["velocity_std"] = df[["velocity_a", "velocity_b", "velocity_c"]].std(axis=1)

    df["expected_midpoint_velocity"] = (df["velocity_a"] + df["velocity_c"]) / 2
    df["midpoint_velocity_deviation"] = (
        df["velocity_b"] - df["expected_midpoint_velocity"]
    )

    if "scenario_id" not in df.columns:
        raise ValueError("scenario_id column is missing before temporal features")

    df = df.sort_values(["scenario_id", "timestep"])

    dt = 0.5

    # First derivatives
    df["dv_dt_a"] = df.groupby("scenario_id")["velocity_a"].diff() / dt
    df["dv_dt_b"] = df.groupby("scenario_id")["velocity_b"].diff() / dt
    df["dv_dt_c"] = df.groupby("scenario_id")["velocity_c"].diff() / dt

    # Second derivatives
    df["d2v_dt2_a"] = df.groupby("scenario_id")["dv_dt_a"].diff() / dt
    df["d2v_dt2_b"] = df.groupby("scenario_id")["dv_dt_b"].diff() / dt
    df["d2v_dt2_c"] = df.groupby("scenario_id")["dv_dt_c"].diff() / dt

    # Fill NaNs
    temporal_cols = [
        "dv_dt_a", "dv_dt_b", "dv_dt_c",
        "d2v_dt2_a", "d2v_dt2_b", "d2v_dt2_c"
    ]
    df[temporal_cols] = df[temporal_cols].fillna(0.0)

    return df


# Run it
df = engineer_velocity_features(df, epsilon)
print(f"Shape after velocity engineering: {df.shape}")

Shape after velocity engineering: (30643, 62)


- Velocity drops at a blockage because the restrication forces flow to slow upstream and starve downstream.
- A leak causes velocity to increase as fluid escapes.
- The midpoint deviation  catches whether node B has a velocity that is abnormal relative to A and C
- Temporal rates catch whether velocity is changing a developing blockage shows progressive velocity decline over consecutive readings.

### Turbulence Features

In [38]:
def engineer_turbulence_features(df, epsilon=1e-8):
    df = df.copy()

    tke_a = np.maximum(df["tke_a"], 0.0)
    tke_b = np.maximum(df["tke_b"], 0.0)
    tke_c = np.maximum(df["tke_c"], 0.0)

    vel_a = df["velocity_a"].clip(lower=epsilon)
    vel_b = df["velocity_b"].clip(lower=epsilon)
    vel_c = df["velocity_c"].clip(lower=epsilon)

    tdr_a = df["tdr_a"].clip(lower=epsilon)
    tdr_b = df["tdr_b"].clip(lower=epsilon)
    tdr_c = df["tdr_c"].clip(lower=epsilon)

    df["turb_intensity_a"] = np.sqrt((2.0 / 3.0) * tke_a) / vel_a
    df["turb_intensity_b"] = np.sqrt((2.0 / 3.0) * tke_b) / vel_b
    df["turb_intensity_c"] = np.sqrt((2.0 / 3.0) * tke_c) / vel_c

    df["tke_drop_ab"] = tke_a - tke_b
    df["tke_drop_bc"] = tke_b - tke_c
    df["tke_drop_ac"] = tke_a - tke_c

    df["tke_tdr_ratio_a"] = tke_a / tdr_a
    df["tke_tdr_ratio_b"] = tke_b / tdr_b
    df["tke_tdr_ratio_c"] = tke_c / tdr_c

    df["turb_length_a"] = (tke_a ** 1.5) / tdr_a
    df["turb_length_b"] = (tke_b ** 1.5) / tdr_b
    df["turb_length_c"] = (tke_c ** 1.5) / tdr_c

    return df
df = engineer_turbulence_features(df, epsilon)
print(f"Shape after turbulence engineering: {df.shape}")

Shape after turbulence engineering: (30643, 72)


- Raw TKE showed near-zero class separation in EDA.
- But turbulence intensity, TKE normalised by velocity, may recover signal because it captures how turbulent the flow is relative to how fast it is moving.
- A fault that creates turbulence without much velocity change will show up in intensity but not in raw TKE.
- The TKE/TDR ratio is the turbulence timescale, how long turbulent eddies survive before dissipating, which changes near fault locations.

### Wall Shear Features

In [39]:
def engineer_wall_shear_features(df, epsilon=1e-8):
    df = df.copy()

    vel_a = df["velocity_a"].clip(lower=epsilon)
    vel_b = df["velocity_b"].clip(lower=epsilon)
    vel_c = df["velocity_c"].clip(lower=epsilon)

    shear_a = df["wall_shear_a"]
    shear_b = df["wall_shear_b"]
    shear_c = df["wall_shear_c"]

    df["shear_drop_ab"] = shear_a - shear_b
    df["shear_drop_bc"] = shear_b - shear_c
    df["shear_drop_ac"] = shear_a - shear_c

    df["shear_vel_ratio_a"] = shear_a / vel_a
    df["shear_vel_ratio_b"] = shear_b / vel_b
    df["shear_vel_ratio_c"] = shear_c / vel_c

    df["total_shear"] = shear_a + shear_b + shear_c
    df["mean_shear"] = df["total_shear"] / 3.0
    df["shear_std"] = df[["wall_shear_a", "wall_shear_b", "wall_shear_c"]].std(axis=1)

    df["shear_asymmetry"] = abs(df["shear_drop_ab"] - df["shear_drop_bc"])

    return df

df = engineer_wall_shear_features(df, epsilon)
print(f"Shape after wall shear engineering: {df.shape}")

Shape after wall shear engineering: (30643, 82)


- Raw wall shear showed a data artifact, Normal reaches 0 while fault classes are floored above 8.
- The shear-to-velocity ratio removes this bias by normalising friction against flow speed, which is the physically meaningful relationship.
- Shear asymmetry catches whether the friction profile is uneven along the pipe, a sign of localised flow disturbance.

### VOF Features

In [40]:
def engineer_vof_features(df, epsilon=1e-8):
    df = df.copy()

    vel_a = df["velocity_a"].clip(lower=epsilon)
    vel_b = df["velocity_b"].clip(lower=epsilon)
    vel_c = df["velocity_c"].clip(lower=epsilon)

    vof_a = df["tailings_vof_a"]
    vof_b = df["tailings_vof_b"]
    vof_c = df["tailings_vof_c"]

    df["vof_drop_ab"] = vof_a - vof_b
    df["vof_drop_bc"] = vof_b - vof_c
    df["vof_drop_ac"] = vof_a - vof_c

    df["mean_vof"] = df[["tailings_vof_a", "tailings_vof_b", "tailings_vof_c"]].mean(axis=1)
    df["vof_std"] = df[["tailings_vof_a", "tailings_vof_b", "tailings_vof_c"]].std(axis=1)

    df["vof_gradient_ac"] = df["vof_drop_ac"] / 40.0
    df["solid_flux_a"] = vof_a * vel_a
    df["solid_flux_b"] = vof_b * vel_b
    df["solid_flux_c"] = vof_c * vel_c

    df["flux_drop_ab"] = df["solid_flux_a"] - df["solid_flux_b"]
    df["flux_drop_bc"] = df["solid_flux_b"] - df["solid_flux_c"]
    df["flux_drop_ac"] = df["solid_flux_a"] - df["solid_flux_c"]

    df["expected_midpoint_vof"] = (vof_a + vof_c) / 2
    df["midpoint_vof_deviation"] = vof_b - df["expected_midpoint_vof"]

    return df

df = engineer_vof_features(df, epsilon)
print(f"Shape after VOF engineering: {df.shape}")

Shape after VOF engineering: (30643, 96)


- Raw VOF per node showed near-complete overlap between classes except at Node A for Blockage.
- The spatial VOF gradient captures the upstream accumulation pattern of blockage,solids pile up upstream while the downstream is starved.
- Solids flux combines concentration and velocity, so it captures the mass flow rate of particles at each node.
- Midpoint VOF deviation catches whether Node B concentration is abnormal — blockage pushes it up, leak pulls it down.

### Cross Features

In [41]:
def engineer_cross_features(df, epsilon=1e-8):
    df = df.copy()

    vel_a = df["velocity_a"].clip(lower=epsilon)
    vel_b = df["velocity_b"].clip(lower=epsilon)
    vel_c = df["velocity_c"].clip(lower=epsilon)

    pres_a = df["node_a_pressure"]
    pres_b = df["node_b_pressure"]
    pres_c = df["node_c_pressure"]

    vof_a = df["tailings_vof_a"]
    vof_b = df["tailings_vof_b"]
    vof_c = df["tailings_vof_c"]

    df["hydraulic_power_a"] = pres_a * vel_a
    df["hydraulic_power_b"] = pres_b * vel_b
    df["hydraulic_power_c"] = pres_c * vel_c

    df["power_loss_ab"] = df["hydraulic_power_a"] - df["hydraulic_power_b"]
    df["power_loss_bc"] = df["hydraulic_power_b"] - df["hydraulic_power_c"]
    df["power_loss_ac"] = df["hydraulic_power_a"] - df["hydraulic_power_c"]

    velocity_epsilon = 0.01  # m/s — minimum meaningful velocity
    df["pv_ratio_b"] = pres_b / (vel_b ** 2 + velocity_epsilon ** 2)

    df["momentum_a"] = (vel_a ** 2) * vof_a
    df["momentum_b"] = (vel_b ** 2) * vof_b
    df["momentum_c"] = (vel_c ** 2) * vof_c

    df["momentum_loss_ab"] = df["momentum_a"] - df["momentum_b"]
    df["momentum_loss_bc"] = df["momentum_b"] - df["momentum_c"]

    return df
df = engineer_cross_features(df, epsilon)
print(f"Shape after cross feature engineering: {df.shape}")

Shape after cross feature engineering: (30643, 108)


- Hydraulic power combines pressure and velocity at each node, it measures the energy flux of the slurry.
- A blockage causes high pressure AND low velocity at Node B simultaneously, so power loss BC becomes very large.
- The pressure-velocity ratio at Node B is essentially a Bernoulli-inspired feature capturing this same simultaneous signature in a single number.
- Momentum flux captures the kinetic energy of the solid-liquid mixture.

### Fault Localization Features

In [42]:
def engineer_localization_features(df, epsilon=1e-8):
    df = df.copy()

    # Pressure-based indicators
    df["leak_pressure_indicator"] = (
        df["midpoint_pressure_deviation"] < -200
    ).astype(int)

    df["blockage_pressure_indicator"] = (
        (df["midpoint_pressure_deviation"] > 500) &
        (df["upstream_downstream_ratio"] > 8.5)
    ).astype(int)

    # Velocity-based indicator
    df["blockage_velocity_indicator"] = (
        df["midpoint_velocity_deviation"] < -0.05
    ).astype(int)

    # VOF-based indicators
    df["leak_vof_indicator"] = (
        df["midpoint_vof_deviation"] < -0.003
    ).astype(int)

    df["blockage_vof_indicator"] = (
        df["midpoint_vof_deviation"] > 0.003
    ).astype(int)

    # Combined scores
    df["leak_score"] = (
        df["leak_pressure_indicator"] +
        df["leak_vof_indicator"]
    )

    df["blockage_score"] = (
        df["blockage_pressure_indicator"] +
        df["blockage_velocity_indicator"] +
        df["blockage_vof_indicator"]
    )

    # Quick validation
    print("Localization indicator validation:")
    print(df.groupby("label")[["leak_score", "blockage_score"]].mean().round(3))

    return df

df = engineer_localization_features(df, epsilon)
print(f"Shape after localization feature engineering: {df.shape}")

Localization indicator validation:
       leak_score  blockage_score
label                            
0           0.664           0.679
1           0.672           1.188
2           0.623           1.181
Shape after localization feature engineering: (30643, 115)


- These binary indicator features directly encode physics-based fault detection rules derived from EDA findings.
- The model will learn weights for these automatically but giving it pre-computed indicators gives it a head start.
- The combined leak and blockage scores summarise multiple weak signals into single numbers that may have higher discriminative value than any individual indicator.

### Rolling Window Features

In [43]:
def engineer_rolling_features(df, window=5):
    df = df.copy()
    columns_to_roll = [
        "node_a_pressure", "node_b_pressure", "node_c_pressure",
        "velocity_b",
        "pressure_drop_bc",
        "midpoint_pressure_deviation",
        "midpoint_velocity_deviation",
        "midpoint_vof_deviation"
    ]

    # Safety: ensure ordering
    if "scenario_id" not in df.columns:
        raise ValueError("scenario_id column is missing before rolling features")

    df = df.sort_values(["scenario_id", "timestep"])

    # Rolling features (SAFE)
    for col in columns_to_roll:
        rolling_group = df.groupby("scenario_id")[col].rolling(window, min_periods=1)

        df[f"rolling_mean_{col}"] = rolling_group.mean().reset_index(level=0, drop=True)
        df[f"rolling_std_{col}"]  = rolling_group.std().reset_index(level=0, drop=True)
    # Fill NaNs (std only)
    rolling_std_cols = [f"rolling_std_{col}" for col in columns_to_roll]
    df[rolling_std_cols] = df[rolling_std_cols].fillna(0.0)

    return df


# Run it
df = engineer_rolling_features(df, window=5)
print(f"Shape after rolling feature engineering: {df.shape}")

Shape after rolling feature engineering: (30643, 131)


- This is the most important cell for LSTM preparation.
- A single reading is ambiguous, it could be noise. But a progressive trend across 5 consecutive readings is a genuine signal.
- Rolling mean captures the underlying trend. Rolling std captures how stable or erratic the signal is, a developing fault shows increasing std as the system becomes unstable
- Grouped by scenario_id to prevent the window crossing scenario boundaries.

### Validate and Save

In [44]:
# Validation
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"Final shape        : {df.shape}")
print(f"Total features     : {df.shape[1] - 3}")  # minus scenario_id, timestep, label
print(f"\nLabel distribution :\n{df['label'].value_counts()}")
print(f"\nNull values        : {df.isnull().sum().sum()}")
print(f"\nInfinite values    : {np.isinf(df.select_dtypes(include=np.number)).sum().sum()}")

# Quick localization check
print("\nBlockage score by class (should be highest for label=2):")
print(df.groupby("label")["blockage_score"].mean().round(3))

print("\nLeak score by class (should be highest for label=1):")
print(df.groupby("label")["leak_score"].mean().round(3))

# Save
output_path = Path.cwd().parent.parent / "data" / "processed" / "engineered_dataset.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

# Save feature names
feature_cols = [c for c in df.columns
                if c not in ["scenario_id", "timestep", "label",
                             "fault_type", "effect_factor"]]
feature_path = Path.cwd().parent.parent / "data" / "processed" / "feature_names.md"
with open(feature_path, "w") as f:
    for feat in feature_cols:
        f.write(feat + "\n")

print(f"Feature names saved to: {feature_path}")
print(f"Total ML features: {len(feature_cols)}")

FEATURE ENGINEERING COMPLETE
Final shape        : (30643, 131)
Total features     : 128

Label distribution :
label
1    10500
0    10500
2     9643
Name: count, dtype: int64

Null values        : 0

Infinite values    : 0

Blockage score by class (should be highest for label=2):
label
0    0.679
1    1.188
2    1.181
Name: blockage_score, dtype: float64

Leak score by class (should be highest for label=1):
label
0    0.664
1    0.672
2    0.623
Name: leak_score, dtype: float64

Saved to: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed/engineered_dataset.csv
Feature names saved to: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed/feature_names.md
Total ML features: 126
